# ViT 안전고리 분류

- 설명: ViT 기반 안전고리 체결 여부 이진 분류 실험입니다.
- 작성자: 조은나라

> 공개 저장소에는 데이터, 모델 가중치, API 키와 노트북 실행 결과를 포함하지 않습니다.


In [ ]:
!unzip -q "/content/분류_test 데이터.zip" -d "/content/분류_test 데이터"
!unzip -q "/content/분류_train 데이터.zip" -d "/content/분류_train 데이터"
!unzip -q "/content/분류_valid 데이터.zip" -d "/content/분류_valid 데이터"

In [ ]:
from pathlib import Path
import shutil

# 기존 unconnected 폴더 경로들
unconnected_dirs = [
    Path("/content/분류_train 데이터/train/unconnected"),
    Path("/content/분류_valid 데이터/분류_valid 데이터/valid/unconnected"),
    Path("/content/분류_test 데이터/test/unconnected"),
]

# 합쳐서 저장할 폴더
save_dir = Path("/content/all_unconnected")
save_dir.mkdir(parents=True, exist_ok=True)

count = 0

for src_dir in unconnected_dirs:
    print("확인 중:", src_dir)

    if not src_dir.exists():
        print("없는 폴더:", src_dir)
        continue

    image_files = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]:
        image_files.extend(src_dir.glob(ext))

    print(f"이미지 수: {len(image_files)}")

    for img_path in image_files:
        # train/valid/test 출처가 겹쳐도 파일명 중복 안 나게 prefix 추가
        split_name = src_dir.parent.name  # train, valid, test
        new_name = f"{split_name}_{img_path.name}"

        dst_path = save_dir / new_name
        shutil.copy2(img_path, dst_path)
        count += 1

print("완료!")
print("총 복사된 unconnected 이미지 수:", count)
print("저장 위치:", save_dir)

In [ ]:
from pathlib import Path
import shutil

# 기존 connected 폴더 경로들
connected_dirs = [
    Path("/content/분류_train 데이터/train/connected"),
    Path("/content/분류_valid 데이터/분류_valid 데이터/valid/connected"),
    Path("/content/분류_test 데이터/test/connected"),
]

# 합쳐서 저장할 폴더
save_dir = Path("/content/all_connected")
save_dir.mkdir(parents=True, exist_ok=True)

count = 0

for src_dir in connected_dirs:
    print("확인 중:", src_dir)

    if not src_dir.exists():
        print("없는 폴더:", src_dir)
        continue

    image_files = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp", "*.JPG", "*.JPEG", "*.PNG"]:
        image_files.extend(src_dir.glob(ext))

    print(f"이미지 수: {len(image_files)}")

    for img_path in image_files:
        # train / valid / test 출처가 겹쳐도 파일명 중복 안 나게 prefix 추가
        split_name = src_dir.parent.name  # train, valid, test
        new_name = f"{split_name}_{img_path.name}"

        dst_path = save_dir / new_name
        shutil.copy2(img_path, dst_path)
        count += 1

print("완료!")
print("총 복사된 connected 이미지 수:", count)
print("저장 위치:", save_dir)

In [ ]:
from pathlib import Path
import shutil
import random

# 원본 connected 전체 폴더
src_dir = Path("/content/all_connected")

# 159장만 저장할 폴더
save_dir = Path("/content/connected_159")
save_dir.mkdir(parents=True, exist_ok=True)

# 이미지 확장자
image_exts = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp", "*.JPG", "*.JPEG", "*.PNG"]

# 이미지 파일 모으기
image_files = []
for ext in image_exts:
    image_files.extend(src_dir.glob(ext))

print("전체 connected 이미지 수:", len(image_files))

# 랜덤 고정
random.seed(42)

# 159장 뽑기
selected_files = random.sample(image_files, 159)

# 복사
for img_path in selected_files:
    dst_path = save_dir / img_path.name
    shutil.copy2(img_path, dst_path)

print("완료!")
print("뽑은 이미지 수:", len(selected_files))
print("저장 위치:", save_dir)

In [ ]:
!pip install -q transformers accelerate evaluate

In [ ]:
import torch
from pathlib import Path
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader
from transformers import ViTForImageClassification, AutoImageProcessor
from torch.optim import AdamW
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

#### 데이터

In [ ]:
!pip install -q scikit-learn

In [ ]:
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split

# 원본 폴더
class_dirs = {
    "connected": Path("/content/connected_159"),
    "unconnected": Path("/content/all_unconnected")
}

# 저장 폴더
save_root = Path("/content/classification_dataset")
save_root.mkdir(parents=True, exist_ok=True)

image_exts = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp", "*.JPG", "*.JPEG", "*.PNG"]

for class_name, src_dir in class_dirs.items():
    # 이미지 파일 목록
    image_files = []
    for ext in image_exts:
        image_files.extend(src_dir.glob(ext))

    image_files = sorted(image_files)

    print(f"{class_name} 전체 이미지 수:", len(image_files))

    # 1차 split: train 70%, temp 30%
    train_files, temp_files = train_test_split(
        image_files,
        test_size=0.30,
        random_state=42,
        shuffle=True
    )

    # 2차 split: temp 30%를 valid 15%, test 15%로 반반 나눔
    valid_files, test_files = train_test_split(
        temp_files,
        test_size=0.50,
        random_state=42,
        shuffle=True
    )

    split_dict = {
        "train": train_files,
        "valid": valid_files,
        "test": test_files
    }

    for split_name, files in split_dict.items():
        dst_dir = save_root / split_name / class_name
        dst_dir.mkdir(parents=True, exist_ok=True)

        for img_path in files:
            shutil.copy2(img_path, dst_dir / img_path.name)

    print("train:", len(train_files))
    print("valid:", len(valid_files))
    print("test :", len(test_files))
    print()

In [ ]:
train_dir = "/content/classification_dataset/train"
valid_dir = "/content/classification_dataset/valid"
test_dir  = "/content/classification_dataset/test"

In [ ]:
model_name = "google/vit-base-patch16-224-in21k"

image_processor = AutoImageProcessor.from_pretrained(model_name)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std
    )
])

In [ ]:
train_dataset = ImageFolder(train_dir, transform=transform)
valid_dataset = ImageFolder(valid_dir, transform=transform)
test_dataset  = ImageFolder(test_dir, transform=transform)

print(train_dataset.classes)
print(train_dataset.class_to_idx)

In [ ]:
batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
num_labels = 2

id2label = {
    0: "connected",
    1: "unconnected"
}

label2id = {
    "connected": 0,
    "unconnected": 1
}

model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.to(device)

#### 학습

In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 30

best_valid_acc = 0
best_model_path = "/content/best_vit_connected_classifier.pt"

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    # train
    model.train()
    train_loss = 0
    train_preds = []
    train_labels = []

    for images, labels in tqdm(train_loader, desc="Training"):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(pixel_values=images, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

    train_acc = accuracy_score(train_labels, train_preds)

    # valid
    model.eval()
    valid_loss = 0
    valid_preds = []
    valid_labels = []

    with torch.no_grad():
        for images, labels in tqdm(valid_loader, desc="Validation"):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(pixel_values=images, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            valid_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            valid_preds.extend(preds.cpu().numpy())
            valid_labels.extend(labels.cpu().numpy())

    valid_acc = accuracy_score(valid_labels, valid_preds)

    print(f"Train Loss: {train_loss / len(train_loader):.4f}")
    print(f"Train Acc : {train_acc:.4f}")
    print(f"Valid Loss: {valid_loss / len(valid_loader):.4f}")
    print(f"Valid Acc : {valid_acc:.4f}")

    # best 모델 저장
    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        torch.save(model.state_dict(), best_model_path)
        print("Best model saved!")

In [ ]:
import shutil
from pathlib import Path

src_path = Path("/content/best_vit_connected_classifier.pt")  # Colab에 저장된 best.pt
dst_dir = Path("/content/drive/MyDrive/vit_connected_classifier")
dst_dir.mkdir(parents=True, exist_ok=True)

dst_path = dst_dir / "best.pt"

shutil.copy2(src_path, dst_path)

print("저장 완료!")
print("저장 위치:", dst_path)

#### test 데이터

In [ ]:
model.load_state_dict(torch.load(best_model_path))
model.to(device)
model.eval()

test_preds = []
test_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(pixel_values=images)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

test_acc = accuracy_score(test_labels, test_preds)

print("Test Accuracy:", test_acc)
print()
print(classification_report(
    test_labels,
    test_preds,
    target_names=test_dataset.classes
))

print("Confusion Matrix")
print(confusion_matrix(test_labels, test_preds))

#### 새 데이터

In [ ]:
from PIL import Image

def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(pixel_values=image)
        probs = torch.softmax(outputs.logits, dim=1)
        pred_idx = torch.argmax(probs, dim=1).item()

    pred_label = id2label[pred_idx]
    confidence = probs[0][pred_idx].item()

    return pred_label, confidence

In [ ]:
image_path = "/content/미체결test2_crop.png"

label, conf = predict_image(image_path)

print("Prediction:", label)
print("Confidence:", conf) # 알고리즘: 후크 객체인식 -> 크롭

In [ ]:
image_path = "/content/미체결test2_crop2.png"

label, conf = predict_image(image_path)

print("Prediction:", label)
print("Confidence:", conf) # 알고리즘: 후크 객체인식 -> 크롭
# ***크롭을 어떻게 하느냐에 따라서 confidence가 많이 달라짐